In [72]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd 


In [73]:
# nous allons vérifier que nous accédons sans soucis à nos identifiants .env
load_dotenv()

CLIENT_ID = os.getenv("FRANCE_TRAVAIL_CLIENT_ID")
CLIENT_SECRET = os.getenv("FRANCE_TRAVAIL_CLIENT_SECRET")

print("Client ID chargé :", CLIENT_ID is not None)
print("Client Secret chargé :", CLIENT_SECRET is not None)

Client ID chargé : True
Client Secret chargé : True


In [74]:
# Construction de la requête OAuth afin de vérifier que notre authentification fonctionne

TOKEN_URL = "https://entreprise.francetravail.fr/connexion/oauth2/access_token?realm=/partenaire"

data = {
    "grant_type" :	"client_credentials" ,
    "client_id"	: CLIENT_ID,
    "client_secret"	: CLIENT_SECRET,
    "scope"	: "api_stats-offres-demandes-emploiv1 offresetdemandesemploi"

}

headers = {
    "Content-Type" : "application/x-www-form-urlencoded"
}

response_token = requests.post(TOKEN_URL, data = data , headers = headers)

if response_token.status_code == 200:
    token_data = response_token.json()
    TOKEN = token_data["access_token"]

    print("Token récupéré :", TOKEN is not None)
    print("Scope obtenu :", token_data.get("scope"))
    print("Expiration :", token_data.get("expires_in"), "secondes")
    
else:
    print("Erreur :", response_token.status_code)
    print(response_token.text)



Token récupéré : True
Scope obtenu : api_stats-offres-demandes-emploiv1 offresetdemandesemploi
Expiration : 1499 secondes


In [75]:
# Vérifions que TOKEN contient maintenant le jeton d'accès

token_data = response_token.json()

TOKEN = token_data["access_token"]

print("Token obtenu :", TOKEN is not None)
print("Type :", token_data["token_type"])
print("Expiration :", token_data["expires_in"], "secondes")

print("Scope du token :", token_data["scope"])
print("Type du token :", token_data["token_type"])


Token obtenu : True
Type : Bearer
Expiration : 1499 secondes
Scope du token : api_stats-offres-demandes-emploiv1 offresetdemandesemploi
Type du token : Bearer


Nous allons commencer par récupérer les données des Endpoints de : 
- Catalogue Indicateur, 
- Offres et 
- Demandeurs.
Ainsi, Avant de collecter des statistiques, nous allons savoir quels indicateurs existent, comment ils sont définis et quels paramètres sont nécessaires pour les récupérer. 
Cela nous évitera de multiplier les appels API sans objectif précis.


In [76]:
# Nous allons appeler l'endpoint Catalogue indicateur de l'API Marché du travail 

response = requests.get(
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/referentiel/details-indicateurs",
    headers={
        "Authorization": f"Bearer {TOKEN}",
        "Accept": "application/json"
    },
    params={
        "codeFamille": "DEMANDEURS",
        "codeIndicateur": "DE_1"
    }
)

print("Status :", response.status_code)
print("URL appelée :", response.url)
print("Réponse :", response.text)

Status : 200
URL appelée : https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/referentiel/details-indicateurs?codeFamille=DEMANDEURS&codeIndicateur=DE_1
Réponse : {"listeIndicateurs":[{"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"ACC_1","libelleIndicateur":"Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)","descriptif":"Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .","codeFamille":"ACCESEMPLOI","disponibiliteIndicateur":1,"listeTypesValeurs":[{"datMaj":"2026-09-23T11:04:31.000+02:00","valeurPrincipale":"O","codeTypeValeur":"NB_ET_PCT","numOrdre":1,"libValeur":"Nombre et % de répartition de ce nombre par caractéristiques.","libelleValeurSpecifique":"Nombre et % des retours à l'emploi AB par durée d'emploi et caractéristiques"},{"

In [77]:

# Pour afficher le résultat de manière à ce qu'il soit lisible

import json

print(json.dumps(response.json(), indent=4, ensure_ascii=False))

{
    "listeIndicateurs": [
        {
            "datMaj": "2026-09-23T11:04:21.000+02:00",
            "codeIndicateur": "ACC_1",
            "libelleIndicateur": "Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)",
            "descriptif": "Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .",
            "codeFamille": "ACCESEMPLOI",
            "disponibiliteIndicateur": 1,
            "listeTypesValeurs": [
                {
                    "datMaj": "2026-09-23T11:04:31.000+02:00",
                    "valeurPrincipale": "O",
                    "codeTypeValeur": "NB_ET_PCT",
                    "numOrdre": 1,
                    "libValeur": "Nombre et % de répartition de ce nombre par caractéristiques.",
                    "libelleValeurSpecifique": "Nom

In [78]:
# Nous allons compter et expliquer les indicateurs que nous avons
## Convertissons la reponse json en objet python

catalogue = response.json()

## Vérifions la structure de la response

print("Clés disponibles: ", catalogue.keys())

## Comptons les indicateurs retournés

indicateurs = catalogue.get("listeIndicateurs", [])

print("Nombres indicateurs: ", len(indicateurs))

## Affichons les premiers indicateurs

for indicateur in indicateurs[:10] :
    print(
        indicateur.get("codeIndicateur"),
        indicateur.get("codeFamille"),
        indicateur.get("libelleIndicateur")
    )

Clés disponibles:  dict_keys(['listeIndicateurs'])
Nombres indicateurs:  19
ACC_1 ACCESEMPLOI Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)
ACC_2 ACCESEMPLOI Nombre et % des accès à l'emploi sorties formation par délai d'accès emploi x durée d'emploi x caractéristiques,Taux d'accès des formés  par caractéristiques
ACC_8 ACCESEMPLOI Nombre et % des retours à l'emploi AB des profils spécifiques x Catégorie Candidat , Taux de retour (= % par rapport aux AB)
CHOM_1 CHOMAGE Taux de chômage
DE_1 DEMANDEURS Nombre et % de DE par catégories x caractéristiques, % par activités (Rome, Compétence)
DE_3 DEMANDEURS Nombre et % de DE sorties formation par caractéristiques, % par activités (Rome, Formation, Compétence)
DE_5 DEMANDEURS Nombre de demandeurs d'emploi enregistrés en entrée (flux) + cumul des 12 derniers mois
DE_8 DEMANDEURS DE avec un profil spécifique par catégories (ABC / Toutes cat

In [79]:
# Nous allons afficher la liste des 19 indicateurs de l'API marché du travail

for indicateur in catalogue["listeIndicateurs"]:
    print("-"*80),
    print( "Code: ", indicateur.get("codeIndicateur")),
    print("Famille", indicateur.get("codeFamille")),
    print("Libéllé", indicateur.get("libelleIndicateur"))


--------------------------------------------------------------------------------
Code:  ACC_1
Famille ACCESEMPLOI
Libéllé Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)
--------------------------------------------------------------------------------
Code:  ACC_2
Famille ACCESEMPLOI
Libéllé Nombre et % des accès à l'emploi sorties formation par délai d'accès emploi x durée d'emploi x caractéristiques,Taux d'accès des formés  par caractéristiques
--------------------------------------------------------------------------------
Code:  ACC_8
Famille ACCESEMPLOI
Libéllé Nombre et % des retours à l'emploi AB des profils spécifiques x Catégorie Candidat , Taux de retour (= % par rapport aux AB)
--------------------------------------------------------------------------------
Code:  CHOM_1
Famille CHOMAGE
Libéllé Taux de chômage
-----------------------------------------------------------------

Cette première version nous permet de recenser les données que nous choisirons pour répondre aux objectifs de notre observatoire de l'emploi. Les indicateurs que nous pourrons sélectionner pour un début sont ceux des offres, des demandeurs d'emploi et des embauches:

- `code:` CHOM_1 `codeFamille :` CHOMAGE et pour `libelleIndicateur: ` Taux de chômage `Utilité pour le projet: ` compléter l'analyse du marché du travail.

- `code:` DE_1 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre et % de DE par catégories x caractéristiques, % par activités (Rome, Compétence);`Utilité pour le projet: ` ce qui nous permettra de répartir les demandeurs d'emploi par catégories et par caractériques.

- `code:` DE_5 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre de demandeurs d'emploi enregistrés en entrée (flux) + cumul des 12 derniers mois;`Utilité pour le projet: ` ce qui nous permettra d'analyser le nombre des demandeurs d'emploi et cumul sur les 12 derniers mois.

-  `code:` DE_9 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre et % de DE ayant retrouvé un emploi selon le metier recherché (FAP87,FAP21);`Utilité pour le projet: ` ce qui nous permettra de connaitre le nombre des demandeurs d'emploi ayant retrouvé un emploi selon le métier recherché.

- `code:` DYN_1 `codeFamille :` DYNEMPLOI et pour `libelleIndicateur: ` DYNEMPLOI Nous n'avons pas encore sa signification exacte, nous l'examinerons si plus amples informations.

- `code:` EMB_1 `codeFamille :` EMBAUCHES et pour `libelleIndicateur: `Nombre et % d'embauches par Catégorie Candidat x durée d'emploi x caractéristiques, % par activités (Rome, Secteur). `Utilité pour le projet: ` Etudier les embauches selon les caractéristiques et les activités.

- `code:` OFF_1 `codeFamille :` OFFRES et pour `libelleIndicateur: ` Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence). `Utilité pour le projet: ` nous allons étudier les offres par térritoire, métier et période.


In [80]:
# Nous allons chercher les informations associées à l'indicateur OFF_1 en affichant sa structure complète

for indicateur in catalogue["listeIndicateurs"]:
    if indicateur.get("codeIndicateur") == "OFF_1" : 
        print(json.dumps(indicateur, indent = 4, ensure_ascii=False))
        break

{
    "datMaj": "2026-09-23T11:04:21.000+02:00",
    "codeIndicateur": "OFF_1",
    "libelleIndicateur": "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)",
    "descriptif": "Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .",
    "codeFamille": "OFFRES",
    "disponibiliteIndicateur": 1,
    "listeTypesValeurs": [
        {
            "datMaj": "2026-09-23T11:04:31.000+02:00",
            "valeurPrincipale": "O",
            "codeTypeValeur": "NB_ET_PCT",
            "numOrdre": 1,
            "libValeur": "Nombre et % de répartition de ce nombre par caractéristiques.",
            "libelleValeurSpecifique": "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) et caractéristiques"
        },
        {
            "datMaj": "2026-09-23T11:04:31.000+02:00",
       

Nous remarquons que OFF_1 contient énormément de paramètres et de combinaisons.

In [81]:
# Nous allons identifier les informations utiles: périodes, territoires, activités disponibles pour OFF_1

## retrouvons la définition complète de OFF_1
off_1 = next(
    item for item in catalogue["listeIndicateurs"]
    if item.get("codeIndicateur") == "OFF_1"
)
# print(off_1)

## Affichons les paramètres utiles pour préparer notre extraction
print("PERIODES")
print(json.dumps(off_1.get("listeTypesPeriodesIndicateur", []), indent =2, ensure_ascii=False))

print("TERRITOIRES")
print(json.dumps(off_1.get("listeTypesTerritoiresIndicateur", []), indent =2, ensure_ascii=False))

print("ACTIVITES")
print(json.dumps(off_1.get("listeTypesActivitesIndicateur", []), indent =2, ensure_ascii=False))

print("NOMMENCLATURES")
print(json.dumps(off_1.get("listeTypesNomenclaturesIndicateur", []), indent =2, ensure_ascii=False))

print("\nNOMBRE DE CROISEMENTS AUTORISÉS")
print(len(off_1.get("listeCroisementsIndicateur", [])))


PERIODES
[
  {
    "datMaj": "2026-09-23T11:04:30.000+02:00",
    "codeTypePeriode": "TRIMESTRE",
    "libelleTypePeriode": "Trimestre",
    "profondeurMax": "12"
  }
]
TERRITOIRES
[
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "NAT",
    "libelleTypeTerritoire": "Niveau National"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "REG",
    "libelleTypeTerritoire": "Région"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "DEP",
    "libelleTypeTerritoire": "Département"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "COM",
    "libelleTypeTerritoire": "Commune"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "BASBMO",
    "libelleTypeTerritoire": "Bassin (BMO)"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "EPCI",
    "libelleTypeTerritoire": "Etablissement public de coopération interco

In [82]:
print("TERRITOIRES")
print(json.dumps(off_1.get("listeTypesTerritoiresIndicateur", []), indent =2, ensure_ascii=False))

TERRITOIRES
[
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "NAT",
    "libelleTypeTerritoire": "Niveau National"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "REG",
    "libelleTypeTerritoire": "Région"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "DEP",
    "libelleTypeTerritoire": "Département"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "COM",
    "libelleTypeTerritoire": "Commune"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "BASBMO",
    "libelleTypeTerritoire": "Bassin (BMO)"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "EPCI",
    "libelleTypeTerritoire": "Etablissement public de coopération intercommunale"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "CLPE",
    "libelleTypeTerritoire": "Comité local pour l'emploi"
  }
]


Nous apprenons ainsi que OFF_1 permet d'avoir:
- Périodes: Trimestres avec une profondeur maximale de 12.
- Territoires: National, Région, Département, Commune, bassin BMO, EPCI et comité local pour l'emploi.
- Activités: toutes activités, métier ROME, secteurs NAF, savoir-être, macro-compétence, filière ...
- Nomenclature: Origine des offres: France travail, partenaires ou toutes les offres.
- Croisement : 252 combinaisons autorisés. Ce sont des combinaisons possibles entre les caractéristiques. Nous allons sélectionner celles qui répondent à notre objectif métier.


Pour Construire un premier jeu de données exploitable,
nous allons commencer par une analyse des offres d'emploi par métier, par région et par trimestre.
Ce qui nous permettra de répondre à des questions concrètes telles que :
- Quels métiers présentent le plus d'offres d'emploi ?
- Comment le volume d'offre évolue t'il au fil des trimestres ?
- Quelles différences observe t-on entre les régions ?
- Les résultats changent-ils selon que l’on considère les offres France Travail ou toutes les offres ?

Nous avons déjà identifier dans les paramètres:
TRIMESTRE pour la période,
REG pour le térritoire région,
ROME pour le métier,
PE et TOFF pour distinguer l'origine des offres.

In [83]:
# Nous allons afficher les 5 premiers croisements et pas les 252

croisements = off_1.get("listeCroisementsIndicateur", [])

print("Nombre total de Croisements :", len(croisements))

for i, croisement in enumerate(croisements[:5], start = 1) : 
    print(f"\n --- Croisement {i} ---")
    print(json.dumps(croisement, indent = 2, ensure_ascii = False))


Nombre total de Croisements : 252

 --- Croisement 1 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF-CUMUL12MOIS"
}

 --- Croisement 2 ---
{
  "datMaj": "2026-09-23T11:04:23.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE"
}

 --- Croisement 3 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF"
}

 --- Croisement 4 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE-C

In [84]:
# Nous observons avec la précédente requête que le codeTypeTerritoire apparait par ordre analphabétique
# Pour vérifier cela, nous allons vérifier la répartition des 252 croisements
# Nous comptons le nombre de croisement par type de territoire

from collections import Counter

repartition = Counter(croisement.get("codeTypeTerritoire")
                        for croisement in croisements
                        )
print("Nombre de croisements par territoire: ")

for territoire, nombre in repartition.items():
    print(f"{territoire} : {nombre}")
    

Nombre de croisements par territoire: 
BASBMO : 36
CLPE : 36
COM : 36
DEP : 36
EPCI : 36
NAT : 36
REG : 36


In [85]:
# Observons dans quel ordre apparait les territoires dans les croisements sans doublons

territoires = [
                croisement.get("codeTypeTerritoire")
                for croisement in croisements
]

territoires_uniques = list(dict.fromkeys(territoires))

print("Ordre des territoires dans la liste :")
print(territoires_uniques)

Ordre des territoires dans la liste :
['BASBMO', 'CLPE', 'COM', 'DEP', 'EPCI', 'NAT', 'REG']


Nous observons de ce fait qu'il y'a 7 territoires et 36 croisements par territoire.

Les sept types de territoires sont :
BASBMO : Bassin BMO
CLPE: Comité local pour l'emploi
COM: Commune
DEP: Département
EPCI: Établissement public de coopération intercommunale
NAT: Niveau national
REG: Région

In [86]:
# Observons les 36 combinaisons d'un territoire - en fonction de RE - région
# Filtrer les croisements de type REG

croisements_reg = [
        croisement
        for croisement in croisements
        if croisement.get("codeTypeTerritoire") == "REG"
]

print("Nombre de croisements régionaux :", len(croisements_reg))

for i, croisement in enumerate(croisements_reg, start=1):
    print(f"\n--- Croisement {i} ---")
    print(json.dumps(croisement, indent=2, ensure_ascii=False))


Nombre de croisements régionaux : 36

--- Croisement 1 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE"
}

--- Croisement 2 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF"
}

--- Croisement 3 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF-CUMUL12MOIS"
}

--- Croisement 4 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE-CUMUL12MOIS"
}

Nous pouvons déduire des 36 croisements que :

Les croisements combinent 9 types d’activité × 4 nomenclatures, toujours avec les mêmes paramètres de période et de territoire. 
Les 9 types d’activité sont CUMUL, ETRE, FILIERE, MACRO_COMP, NAF21, NAF5, NAF88, OBJECTIF et ROME.

Les quatre valeurs de nomenclature sont :

PE : offres France Travail,
PE-CUMUL12MOIS : cumul sur les 12 derniers mois des offres France Travail,
TOFF : toutes les offres,
TOFF-CUMUL12MOIS : cumul sur les 12 derniers mois de toutes les offres.


In [87]:

import requests

url = (
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres"
)

payload = {
    "codeTypeTerritoire": "REG",
    "codeTerritoire": "75",
    "codeTypeActivite": "ROME",
    "codeActivite": "A1203",
    "codeTypePeriode": "TRIMESTRE",
    "codeTypeNomenclature": "ORIGINEOFF",
    "codeNomenclature": "TOFF"
}

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(
    url,
    json=payload,
    headers=headers
)

print("Statut :", response.status_code)
print("Type de contenu :", response.headers.get("Content-Type"))
print("Réponse :", response.text[:3000])

Statut : 200
Type de contenu : application/json
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)","listeValeursParPeriode":[{"datMaj":"2026-09-23T09:06:21.000+02:00","codeTypeTerritoire":"REG","codeTerritoire":"75","libTerritoire":"NOUVELLE-AQUITAINE","codeTypeActivite":"ROME","codeActivite":"A1203","libActivite":"Agent / Agente d'entretien des espaces verts","codeNomenclature":"PE","libNomenclature":"Offres France Travail","codeTypePeriode":"TRIMESTRE","codePeriode":"2026T2","libPeriode":"2ème trimestre 2026","valeurPrincipaleNombre":290,"valeurSecondairePourcentage":0.0,"listeValeurParCaract":[{"codeTypeCaract":"TYPECTR","codeCaract":"CDI","libCaract":"CDI","nombre":60,"pourcentage":23.0,"masque":false},{"codeTypeCaract":"TYPECTR","codeCaract":"CDD1","libCaract":"CDD < 1 mois","n

Nous allons transformer notre résultat JSON en Dataframe.


In [88]:
resultat = response.json()

# Extraire les statistiques par périodes
valeurs = resultat.get("listeValeursParPeriode", [])

df_offres = pd.json_normalize(valeurs)

display(df_offres.head())



,datMaj,codeTypeTerritoire,codeTerritoire,libTerritoire,codeTypeActivite,codeActivite,libActivite,codeNomenclature,libNomenclature,codeTypePeriode,codePeriode,libPeriode,valeurPrincipaleNombre,valeurSecondairePourcentage,listeValeurParCaract
0,2026-09-23T09:06:21.000+02:00,REG,75,NOUVELLE-AQUITAINE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE,Offres France Travail,TRIMESTRE,2026T2,2ème trimestre 2026,290,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
1,2026-09-23T09:06:21.000+02:00,REG,75,NOUVELLE-AQUITAINE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE-CUMUL12MOIS,Cumul 12 derniers mois offres France Travail,TRIMESTRE,2026T2,2ème trimestre 2026,890,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
2,2026-09-23T09:06:23.000+02:00,REG,75,NOUVELLE-AQUITAINE,ROME,A1203,Agent / Agente d'entretien des espaces verts,TOFF,Toutes offres,TRIMESTRE,2026T2,2ème trimestre 2026,390,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
3,2026-09-23T09:06:22.000+02:00,REG,75,NOUVELLE-AQUITAINE,ROME,A1203,Agent / Agente d'entretien des espaces verts,TOFF-CUMUL12MOIS,Cumul 12 derniers mois toutes offres,TRIMESTRE,2026T2,2ème trimestre 2026,1160,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
4,2026-09-23T09:06:21.000+02:00,REG,75,NOUVELLE-AQUITAINE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE,Offres France Travail,TRIMESTRE,2026T1,1er trimestre 2026,260,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."


In [89]:
# informations de notre df_offres

df_offres.info()

<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   datMaj                       88 non-null     str    
 1   codeTypeTerritoire           88 non-null     str    
 2   codeTerritoire               88 non-null     str    
 3   libTerritoire                88 non-null     str    
 4   codeTypeActivite             88 non-null     str    
 5   codeActivite                 88 non-null     str    
 6   libActivite                  88 non-null     str    
 7   codeNomenclature             88 non-null     str    
 8   libNomenclature              88 non-null     str    
 9   codeTypePeriode              88 non-null     str    
 10  codePeriode                  88 non-null     str    
 11  libPeriode                   88 non-null     str    
 12  valeurPrincipaleNombre       88 non-null     int64  
 13  valeurSecondairePourcentage  88 n

In [90]:
# Liste des colonnes de df_offres

df_offres.columns


Index(['datMaj', 'codeTypeTerritoire', 'codeTerritoire', 'libTerritoire',
       'codeTypeActivite', 'codeActivite', 'libActivite', 'codeNomenclature',
       'libNomenclature', 'codeTypePeriode', 'codePeriode', 'libPeriode',
       'valeurPrincipaleNombre', 'valeurSecondairePourcentage',
       'listeValeurParCaract'],
      dtype='str')

In [91]:
# Nous allons drop les colonnes qui ne nous sommes pas principales

df_offres = df_offres.drop(columns= ['datMaj', 'listeValeurParCaract'])

In [92]:
# Nos colonnes principales sont:
df_offres.info()

<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   codeTypeTerritoire           88 non-null     str    
 1   codeTerritoire               88 non-null     str    
 2   libTerritoire                88 non-null     str    
 3   codeTypeActivite             88 non-null     str    
 4   codeActivite                 88 non-null     str    
 5   libActivite                  88 non-null     str    
 6   codeNomenclature             88 non-null     str    
 7   libNomenclature              88 non-null     str    
 8   codeTypePeriode              88 non-null     str    
 9   codePeriode                  88 non-null     str    
 10  libPeriode                   88 non-null     str    
 11  valeurPrincipaleNombre       88 non-null     int64  
 12  valeurSecondairePourcentage  88 non-null     float64
dtypes: float64(1), int64(1), str(11)


df_offres contient :

- 88 lignes : 88 enregistrements statistiques retournés par l'API.

- 13 colonnes : les informations descriptives et les valeurs statistiques principales.

- Aucune valeur manquante dans ces 13 colonnes.

- 2 colonnes numériques : valeurPrincipaleNombre et valeurSecondairePourcentage.

In [93]:
# Nous allons faire une observation des données de ce dataframe

# Nombre de résultats par nomenclature
print(df_offres["codeNomenclature"].value_counts())

# Nombre de résultats par Période

print(df_offres["codePeriode"].value_counts().sort_index())



codeNomenclature
PE                  22
PE-CUMUL12MOIS      22
TOFF                22
TOFF-CUMUL12MOIS    22
Name: count, dtype: int64
codePeriode
2021T1    4
2021T2    4
2021T3    4
2021T4    4
2022T1    4
2022T2    4
2022T3    4
2022T4    4
2023T1    4
2023T2    4
2023T3    4
2023T4    4
2024T1    4
2024T2    4
2024T3    4
2024T4    4
2025T1    4
2025T2    4
2025T3    4
2025T4    4
2026T1    4
2026T2    4
Name: count, dtype: int64


In [94]:
# Vérification des combinaisons de territoires et de métiers

print("Territoires :", df_offres["libTerritoire"].nunique())

print("Territoires :", df_offres["libTerritoire"].unique())

print("Periodes :", df_offres["codeActivite"].nunique())

print("Periodes :", df_offres["codeActivite"].unique())



Territoires : 1
Territoires : <StringArray>
['NOUVELLE-AQUITAINE']
Length: 1, dtype: str
Periodes : 1
Periodes : <StringArray>
['A1203']
Length: 1, dtype: str


L’API renvoie les données du métier A1203, en Nouvelle-Aquitaine, sur 22 trimestres, de 2021T1 à 2026T2, pour quatre nomenclatures :

PE : offres France Travail,
PE-CUMUL12MOIS : cumul sur 12 mois des offres France Travail,
TOFF : toutes les offres,
TOFF-CUMUL12MOIS : cumul sur 12 mois de toutes les offres.

In [95]:
df_offres.columns

Index(['codeTypeTerritoire', 'codeTerritoire', 'libTerritoire',
       'codeTypeActivite', 'codeActivite', 'libActivite', 'codeNomenclature',
       'libNomenclature', 'codeTypePeriode', 'codePeriode', 'libPeriode',
       'valeurPrincipaleNombre', 'valeurSecondairePourcentage'],
      dtype='str')

In [96]:
# Nous allons comparer les 4 nomenclatures sur les derniers trimestres

colonnes = [
            'codePeriode',
            'libPeriode',
            'codeNomenclature',
            'libNomenclature',
            'valeurPrincipaleNombre', 
            'valeurSecondairePourcentage'
        ]

display(df_offres.loc[
            df_offres["codePeriode"].isin(["2026T1" , "2026T2"]),
            colonnes
].sort_values(["codePeriode", "codeNomenclature"])

)


,codePeriode,libPeriode,codeNomenclature,libNomenclature,valeurPrincipaleNombre,valeurSecondairePourcentage
4,2026T1,1er trimestre 2026,PE,Offres France Travail,260,0.0
5,2026T1,1er trimestre 2026,PE-CUMUL12MOIS,Cumul 12 derniers mois offres France Travail,930,0.0
6,2026T1,1er trimestre 2026,TOFF,Toutes offres,360,0.0
7,2026T1,1er trimestre 2026,TOFF-CUMUL12MOIS,Cumul 12 derniers mois toutes offres,1190,0.0
0,2026T2,2ème trimestre 2026,PE,Offres France Travail,290,0.0
1,2026T2,2ème trimestre 2026,PE-CUMUL12MOIS,Cumul 12 derniers mois offres France Travail,890,0.0
2,2026T2,2ème trimestre 2026,TOFF,Toutes offres,390,0.0
3,2026T2,2ème trimestre 2026,TOFF-CUMUL12MOIS,Cumul 12 derniers mois toutes offres,1160,0.0


Pour le métier Agent/Agente d’entretien des espaces verts (ROME A1203) en Nouvelle-Aquitaine, on observe les variations.
Les valeurs du cumul sur 12 mois diminuent alors que les valeurs trimestrielles augmentent.


In [97]:
df_offres.columns

Index(['codeTypeTerritoire', 'codeTerritoire', 'libTerritoire',
       'codeTypeActivite', 'codeActivite', 'libActivite', 'codeNomenclature',
       'libNomenclature', 'codeTypePeriode', 'codePeriode', 'libPeriode',
       'valeurPrincipaleNombre', 'valeurSecondairePourcentage'],
      dtype='str')

In [98]:
# Vérifions maintenant que les quatre nomenclatures sont présentes pour toutes les périodes et qu'il n'y a pas de doublons inattendus.
cols = [
            'codeTerritoire',
            'codeActivite',
            'codePeriode',
            'codeNomenclature',
            'valeurPrincipaleNombre', 
            'valeurSecondairePourcentage'
        ]
print("Nombres de doublons :", df_offres.duplicated(subset=cols).sum())

# Vérifions le nombre de lignes par trimestre

print("Nombres de lignes par trimestre :", df_offres.groupby("codePeriode").size().sort_index())

# Vérifier les valeurs manquantes

print(df_offres.isna().sum())


Nombres de doublons : 0
Nombres de lignes par trimestre : codePeriode
2021T1    4
2021T2    4
2021T3    4
2021T4    4
2022T1    4
2022T2    4
2022T3    4
2022T4    4
2023T1    4
2023T2    4
2023T3    4
2023T4    4
2024T1    4
2024T2    4
2024T3    4
2024T4    4
2025T1    4
2025T2    4
2025T3    4
2025T4    4
2026T1    4
2026T2    4
dtype: int64
codeTypeTerritoire             0
codeTerritoire                 0
libTerritoire                  0
codeTypeActivite               0
codeActivite                   0
libActivite                    0
codeNomenclature               0
libNomenclature                0
codeTypePeriode                0
codePeriode                    0
libPeriode                     0
valeurPrincipaleNombre         0
valeurSecondairePourcentage    0
dtype: int64


Nos observations montrent que :
- absence de doublons
- absence de valeurs manquantes
- Répartition par trimestre : 4 lignes avec une ligne par nomenclature et par trimestre
- Nombre de lignes 88 avec 22 trimestres × 4 nomenclatures.


Pour notre projet, nous allons commencer avec l'indicateur OFF_1 (offres d'emploi), en conservant les quatre nomenclatures et les trimestres disponibles. 
Nous pourrons ensuite élargir la collecte à plusieurs métiers et territoires.

In [99]:

# Nous allons identifier les combinaisons disponibles pour la France métropolitaine les croisements de l'indicateur OFF_1.
# Examinons les types de territoires disponibles
display(off_1["listeTypesTerritoiresIndicateur"])

# Examinons les types d'activités disponibles
display(off_1["listeTypesActivitesIndicateur"])

# Examinons les nomenclatures disponibles
display(off_1["listeTypesNomenclaturesIndicateur"])


[{'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'NAT',
  'libelleTypeTerritoire': 'Niveau National'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'REG',
  'libelleTypeTerritoire': 'Région'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'DEP',
  'libelleTypeTerritoire': 'Département'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'COM',
  'libelleTypeTerritoire': 'Commune'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'libelleTypeTerritoire': 'Bassin (BMO)'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'EPCI',
  'libelleTypeTerritoire': 'Etablissement public de coopération intercommunale'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'CLPE',
  'libelleTypeTerritoire': "Comité local pour l'emploi"}]

[{'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'CUMUL',
  'libelleTypeActivite': 'Cumul (toutes activités)'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'ETRE',
  'libelleTypeActivite': 'Savoir-être'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'MACRO_COMP',
  'libelleTypeActivite': 'Macro compétence'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'NAF21',
  'libelleTypeActivite': "Secteur d'activité (NAF21)"},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'NAF5',
  'libelleTypeActivite': "Secteur d'activité (NAF5)"},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'NAF88',
  'libelleTypeActivite': "Secteur d'activité (NAF88)"},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'OBJECTIF',
  'libelleTypeActivite': 'Objectif'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeActivite': 'ROME',
  'libelleTypeActivite': 'Métier (ROME)'}

[{'datMaj': '2026-09-23T11:04:12.000+02:00',
  'codeTypeNomenclature': 'ORIGINEOFF',
  'libelleTypeNomenclature': "Origine des offres d'emploi (PE, Partenaire .)",
  'listeValeursNomenclaturesIndicateur': [{'datMaj': '2026-09-23T11:04:31.000+02:00',
    'valeurNomenclature': 'PE',
    'libelleNomenclature': 'Offres France Travail'},
   {'datMaj': '2026-09-23T11:04:31.000+02:00',
    'valeurNomenclature': 'PE-CUMUL12MOIS',
    'libelleNomenclature': 'Cumul 12 derniers mois offres France Travail'},
   {'datMaj': '2026-09-23T11:04:31.000+02:00',
    'valeurNomenclature': 'TOFF',
    'libelleNomenclature': 'Toutes offres'},
   {'datMaj': '2026-09-23T11:04:31.000+02:00',
    'valeurNomenclature': 'TOFF-CUMUL12MOIS',
    'libelleNomenclature': 'Cumul 12 derniers mois toutes offres'}]}]

OFF_1 propose 7 niveaux géographiques mais pour notre première collecte, nous allons nous concentrer sur NAT et REG, afin de commencer par une vue nationale puis de comparer les régions.


In [100]:

# Nous allons afficher les types d'activités autorisés pour OFF_1
for activite in off_1["listeTypesActivitesIndicateur"]:
    print(
        activite.get("codeTypeActivite"),
        "-",
        activite.get("libelleTypeActivite")
    )

CUMUL - Cumul (toutes activités)
ETRE - Savoir-être
MACRO_COMP - Macro compétence
NAF21 - Secteur d'activité (NAF21)
NAF5 - Secteur d'activité (NAF5)
NAF88 - Secteur d'activité (NAF88)
OBJECTIF - Objectif
ROME - Métier (ROME)
FILIERE - Filière d'activité


Nous allons priviligier pour notre collecte les activités de:

- ROME qui nous permettra de classer les métiers selon le nombre d'offres

- NAF21 nous permettra d'identifier les secteurs d'activité qui concentrent le plus d'offres

- CUMUL pourra nous aider à disposer d'une vue globale


In [101]:

croisements = off_1["listeCroisementsIndicateur"]

print("Nombre total de croisements :", len(croisements))
print("\nClés disponibles :")
print(croisements[0].keys())

display(croisements[:5])

Nombre total de croisements : 252

Clés disponibles :
dict_keys(['datMaj', 'codeTypeTerritoire', 'codeTypeActivite', 'codeTypePeriode', 'codeTypeNomenclature', 'codeNomenclature'])


[{'datMaj': '2026-09-23T11:04:22.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'codeTypeActivite': 'CUMUL',
  'codeTypePeriode': 'TRIMESTRE',
  'codeTypeNomenclature': 'ORIGINEOFF',
  'codeNomenclature': 'TOFF-CUMUL12MOIS'},
 {'datMaj': '2026-09-23T11:04:23.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'codeTypeActivite': 'CUMUL',
  'codeTypePeriode': 'TRIMESTRE',
  'codeTypeNomenclature': 'ORIGINEOFF',
  'codeNomenclature': 'PE'},
 {'datMaj': '2026-09-23T11:04:22.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'codeTypeActivite': 'CUMUL',
  'codeTypePeriode': 'TRIMESTRE',
  'codeTypeNomenclature': 'ORIGINEOFF',
  'codeNomenclature': 'TOFF'},
 {'datMaj': '2026-09-23T11:04:22.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'codeTypeActivite': 'CUMUL',
  'codeTypePeriode': 'TRIMESTRE',
  'codeTypeNomenclature': 'ORIGINEOFF',
  'codeNomenclature': 'PE-CUMUL12MOIS'},
 {'datMaj': '2026-09-23T11:04:22.000+02:00',
  'codeTypeTerritoire': 'BASBMO',
  'codeTypeActivite': 'ETRE',
  'codeType

In [102]:
# Convertir les croisements en DataFrame
df_croisements = pd.DataFrame(croisements)

# Types d'activités retenus pour notre première collecte
activites_cibles = ["ROME", "NAF21", "CUMUL"]

# Nomenclatures utiles pour comparer les offres
nomenclatures_cibles = ["PE", "TOFF"]

# Filtrer les combinaisons utiles
df_plan_collecte = df_croisements[
    df_croisements["codeTypeTerritoire"].isin(["NAT", "REG"])
    & df_croisements["codeTypeActivite"].isin(activites_cibles)
    & df_croisements["codeNomenclature"].isin(nomenclatures_cibles)
].copy()

print("Nombre de croisements retenus :", len(df_plan_collecte))

display(
    df_plan_collecte
    .groupby(
        ["codeTypeTerritoire", "codeTypeActivite", "codeNomenclature"]
    )
    .size()
    .rename("nombre")
    .reset_index()
)

Nombre de croisements retenus : 12


,codeTypeTerritoire,codeTypeActivite,codeNomenclature,nombre
0,NAT,CUMUL,PE,1
1,NAT,CUMUL,TOFF,1
2,NAT,NAF21,PE,1
3,NAT,NAF21,TOFF,1
4,NAT,ROME,PE,1
5,NAT,ROME,TOFF,1
6,REG,CUMUL,PE,1
7,REG,CUMUL,TOFF,1
8,REG,NAF21,PE,1
9,REG,NAF21,TOFF,1


In [103]:

display(
    df_plan_collecte[
        df_plan_collecte["codeTypeActivite"].isin(["ROME", "NAF21"])
    ].sort_values(
        ["codeTypeTerritoire", "codeTypeActivite", "codeNomenclature"]
    )
)

,datMaj,codeTypeTerritoire,codeTypeActivite,codeTypePeriode,codeTypeNomenclature,codeNomenclature
197,2026-09-23T11:04:22.000+02:00,NAT,NAF21,TRIMESTRE,ORIGINEOFF,PE
196,2026-09-23T11:04:22.000+02:00,NAT,NAF21,TRIMESTRE,ORIGINEOFF,TOFF
212,2026-09-23T11:04:23.000+02:00,NAT,ROME,TRIMESTRE,ORIGINEOFF,PE
214,2026-09-23T11:04:23.000+02:00,NAT,ROME,TRIMESTRE,ORIGINEOFF,TOFF
232,2026-09-23T11:04:22.000+02:00,REG,NAF21,TRIMESTRE,ORIGINEOFF,PE
235,2026-09-23T11:04:23.000+02:00,REG,NAF21,TRIMESTRE,ORIGINEOFF,TOFF
251,2026-09-23T11:04:22.000+02:00,REG,ROME,TRIMESTRE,ORIGINEOFF,PE
249,2026-09-23T11:04:22.000+02:00,REG,ROME,TRIMESTRE,ORIGINEOFF,TOFF


In [104]:

# Afficher un croisement complet de type ROME au niveau national
croisement_rome_nat = next(
    c for c in croisements
    if c["codeTypeTerritoire"] == "NAT"
    and c["codeTypeActivite"] == "ROME"
    and c["codeNomenclature"] == "PE"
)

display(croisement_rome_nat)

{'datMaj': '2026-09-23T11:04:23.000+02:00',
 'codeTypeTerritoire': 'NAT',
 'codeTypeActivite': 'ROME',
 'codeTypePeriode': 'TRIMESTRE',
 'codeTypeNomenclature': 'ORIGINEOFF',
 'codeNomenclature': 'PE'}

 Pour notre projet, le référentiel des métiers est essentiel : il nous permettra d'obtenir les codes et les libellés nécessaires pour préparer les requêtes statistiques.
 

In [105]:

# Nous allons appeler l'API ROME 4.0 - Métiers et obtenir un jeton OAuth pour ROME 4.0
TOKEN_URL = (
    "https://entreprise.francetravail.fr/"
    "connexion/oauth2/access_token?realm=/partenaire"
)

data_rome = {
    "grant_type": "client_credentials",
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "scope": "api_rome-metiersv1 nomenclatureRome"
}

headers_token = {
    "Content-Type": "application/x-www-form-urlencoded"
}

response_token_rome = requests.post(
    TOKEN_URL,
    data=data_rome,
    headers=headers_token,
    timeout=30
)

print("Statut OAuth ROME :", response_token_rome.status_code)

if response_token_rome.status_code == 200:
    token_data_rome = response_token_rome.json()
    TOKEN_ROME = token_data_rome["access_token"]

    print("Token récupéré :", bool(TOKEN_ROME))
    print("Scope retourné :", token_data_rome.get("scope"))
else:
    print("Réponse :", response_token_rome.text[:1000])

Statut OAuth ROME : 200
Token récupéré : True
Scope retourné : api_rome-metiersv1 nomenclatureRome


In [106]:
# Testons l'endpoint: liste des métiers

url_rome = (
    "https://api.francetravail.io/partenaire/rome-metiers/v1/metiers/metier"
)

headers_rome = {
    "Authorization": f"Bearer {TOKEN_ROME}",
    "Accept": "application/json"
}

response_rome = requests.get(
    url_rome,
    headers=headers_rome,
    timeout=30
)

print("Statut API ROME :", response_rome.status_code)
print("Type de contenu :", response_rome.headers.get("Content-Type"))

if response_rome.status_code == 200:
    donnees_rome = response_rome.json()

    print("Type de réponse :", type(donnees_rome))

    if isinstance(donnees_rome, dict):
        print("Clés disponibles :", donnees_rome.keys())
        display(donnees_rome)
    elif isinstance(donnees_rome, list):
        print("Nombre d'éléments retournés :", len(donnees_rome))
        display(donnees_rome[:5])
else:
    print("Réponse :", response_rome.text[:1500])

Statut API ROME : 200
Type de contenu : application/json
Type de réponse : <class 'list'>
Nombre d'éléments retournés : 1923


[{'code': 'N2215', 'libelle': 'Coordonnateur / Coordonnatrice zone avion'},
 {'code': 'N4207',
  'libelle': 'Régulateur / Régulatrice poste central transport sanitaire de personnes'},
 {'code': 'L1321',
  'libelle': 'Directeur / Directrice de production de spectacles'},
 {'code': 'I1639', 'libelle': "Contrôleur / Contrôleuse de moteurs d'avions"},
 {'code': 'N2217',
  'libelle': "Chef d'équipe contrôleur aérien / Cheffe d'équipe contrôleuse aérienne"}]

Nos deux API marché du travail et ROME 4.0 Métiers répondent correctement.

Elles nous permettront de relier les statistiques aux informations du référentiel métier lorsque les codes le permettent. Cela nous aidera à rendre les résultats plus compréhensibles et à analyser les tendances par métier et par territoire.


In [107]:
# Nous regardons la structure des données ROME
resultat_rome = response_rome.json()
print(resultat_rome)


[{'code': 'N2215', 'libelle': 'Coordonnateur / Coordonnatrice zone avion'}, {'code': 'N4207', 'libelle': 'Régulateur / Régulatrice poste central transport sanitaire de personnes'}, {'code': 'L1321', 'libelle': 'Directeur / Directrice de production de spectacles'}, {'code': 'I1639', 'libelle': "Contrôleur / Contrôleuse de moteurs d'avions"}, {'code': 'N2217', 'libelle': "Chef d'équipe contrôleur aérien / Cheffe d'équipe contrôleuse aérienne"}, {'code': 'K1924', 'libelle': 'Clerc de commissaire de justice'}, {'code': 'K1923', 'libelle': 'Clerc de commissaire-priseur'}, {'code': 'K1231', 'libelle': 'Patient partenaire / Patiente partenaire en santé'}, {'code': 'K1429', 'libelle': "Administrateur / Administratrice de l'Etat"}, {'code': 'K1428', 'libelle': 'Conseiller / Conseillère de cabinet ministériel'}, {'code': 'K2515', 'libelle': 'Agent / Agente cynotechnique en détection des explosifs'}, {'code': 'I1411', 'libelle': 'Agent / Agente de reconditionnement numérique'}, {'code': 'C1124', 

In [108]:
# Pour avoir une observation de structure plus lisible

print(json.dumps(resultat_rome, indent=4, ensure_ascii=False))


[
    {
        "code": "N2215",
        "libelle": "Coordonnateur / Coordonnatrice zone avion"
    },
    {
        "code": "N4207",
        "libelle": "Régulateur / Régulatrice poste central transport sanitaire de personnes"
    },
    {
        "code": "L1321",
        "libelle": "Directeur / Directrice de production de spectacles"
    },
    {
        "code": "I1639",
        "libelle": "Contrôleur / Contrôleuse de moteurs d'avions"
    },
    {
        "code": "N2217",
        "libelle": "Chef d'équipe contrôleur aérien / Cheffe d'équipe contrôleuse aérienne"
    },
    {
        "code": "K1924",
        "libelle": "Clerc de commissaire de justice"
    },
    {
        "code": "K1923",
        "libelle": "Clerc de commissaire-priseur"
    },
    {
        "code": "K1231",
        "libelle": "Patient partenaire / Patiente partenaire en santé"
    },
    {
        "code": "K1429",
        "libelle": "Administrateur / Administratrice de l'Etat"
    },
    {
        "code": "K1428",
 

In [109]:
# Afficher le premier métier retourné par l'API
if len(resultat_rome) > 0:
    print(resultat_rome[0])

{'code': 'N2215', 'libelle': 'Coordonnateur / Coordonnatrice zone avion'}


In [110]:
# Identifier les champs disponibles
if len(resultat_rome) > 0:
    print(resultat_rome[0].keys())


dict_keys(['code', 'libelle'])


## le reférentiel ROME

Notre première collecte : enrégistrer le reférentiel ROME


In [111]:
# Nous allons récupérer les données ROME et les transformer en Dataframe puis contrôler leurs qualité avant de les inclure dans le projet

# Transformons la liste des métiers resultat_rome en Dataframe
df_metiers_rome = pd.DataFrame(resultat_rome)

# Vérifions la structure
display(df_metiers_rome.head())

# Nous allons vérifier les dimensions
print(f"le nombre de lignes :", len(df_metiers_rome))

print(df_metiers_rome.info()) # Nous avons 2 colonnes et 1923 lignes 

print(df_metiers_rome.columns.tolist())


,code,libelle
0,N2215,Coordonnateur / Coordonnatrice zone avion
1,N4207,Régulateur / Régulatrice poste central transpo...
2,L1321,Directeur / Directrice de production de specta...
3,I1639,Contrôleur / Contrôleuse de moteurs d'avions
4,N2217,Chef d'équipe contrôleur aérien / Cheffe d'équ...


le nombre de lignes : 1923
<class 'pandas.DataFrame'>
RangeIndex: 1923 entries, 0 to 1922
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   code     1923 non-null   str  
 1   libelle  1923 non-null   str  
dtypes: str(2)
memory usage: 30.2 KB
None
['code', 'libelle']


In [112]:
# Nous allons contrôler la qualité de nos données et si les codes sont repétés
print("Valeurs manquantes :")
print(df_metiers_rome.isna().sum())

print("Nombres de doublons :")
print(df_metiers_rome.duplicated().sum())

print("\nCode métier doublon : ")
print(df_metiers_rome["code"].duplicated().sum())

print("\nCode manquant : ")
print(df_metiers_rome["code"].isna().sum())

print("\nLibelle manquant : ")
print(df_metiers_rome["libelle"].isna().sum())


Valeurs manquantes :
code       0
libelle    0
dtype: int64
Nombres de doublons :
0

Code métier doublon : 
0

Code manquant : 
0

Libelle manquant : 
0


In [113]:
df_metiers_rome.head()

,code,libelle
0,N2215,Coordonnateur / Coordonnatrice zone avion
1,N4207,Régulateur / Régulatrice poste central transpo...
2,L1321,Directeur / Directrice de production de specta...
3,I1639,Contrôleur / Contrôleuse de moteurs d'avions
4,N2217,Chef d'équipe contrôleur aérien / Cheffe d'équ...


In [114]:
# Nos données sont satisfaisantes, nous allons les charger
# Conserver la réponse originale de l'API

with open("data/raw/rome_metiers.json",
            "w",
            encoding = "utf-8",
            ) as fichier:
                json.dump(resultat_rome, fichier, ensure_ascii=False,indent=2)


In [115]:
# Enregistrer le dataframe

df_metiers_rome.to_csv(
                    "data/processed/rome_metiers.csv",
                    index=False,
                    encoding = "utf-8-sig"
            )



## API marché du travail
### Collecte des offres d'emploi

Nous avons défini le périmètre de notre collecte des offres d'emploi:

Territoire : France métropolitaine

Période : du premier trimestre 2023 au dernier trimestre disponible en 2026

Métiers : ensemble des métiers disponibles dans le référentiel ROME.

Secteurs : nomenclature d’activité disponible dans l’API, notamment NAF21.

Statistiques : offres d’emploi, en distinguant les offres France Travail et toutes les offres lorsque les données le permettent.

Avant de lancer une collecte à grande échelle, nous devons vérifier le nombre d'appels nécessaires et les paramètres géographiques, pour éviter une collecte trop volumineuse ou des données qui dépasseraient notre périmètre.



In [116]:
# Préparer la collecte offres d'emploi

# Nous allons repartir depuis notre df_offres qui étéit un test d'appel de l'api pour la région  75 aquitaine et codeActivite : A1203
# Nous allons vérifier les informations déjà présentes dans df_offres afin de définir notre périmètre pour notre collecte.

print("Dimensions : ", df_offres.shape)

print("Lister les colonnes : ", df_offres.columns.tolist())

print("Territoires : ", df_offres["codeTypeTerritoire"].unique())

print("Types Activités : ", df_offres["codeTypeActivite"].unique())

print("Types Nomenclatures : ", df_offres["codeNomenclature"].unique())

print("Périodes : ", df_offres["codePeriode"].min(), "à" , df_offres["codePeriode"].max() )


Dimensions :  (88, 13)
Lister les colonnes :  ['codeTypeTerritoire', 'codeTerritoire', 'libTerritoire', 'codeTypeActivite', 'codeActivite', 'libActivite', 'codeNomenclature', 'libNomenclature', 'codeTypePeriode', 'codePeriode', 'libPeriode', 'valeurPrincipaleNombre', 'valeurSecondairePourcentage']
Territoires :  <StringArray>
['REG']
Length: 1, dtype: str
Types Activités :  <StringArray>
['ROME']
Length: 1, dtype: str
Types Nomenclatures :  <StringArray>
['PE', 'PE-CUMUL12MOIS', 'TOFF', 'TOFF-CUMUL12MOIS']
Length: 4, dtype: str
Périodes :  2021T1 à 2026T2


In [117]:
# Vérifions so nous pouvons limiter la période de 2023T1 à 2026T2 pour notre collecte
# Filtrer la période d'étude

df_offres_periodes = df_offres[df_offres["codePeriode"].between("2023T1" , "2026T2 ")].copy()

print("Dimensions à filtrage : ", df_offres_periodes.shape)

print("Périodes conservées : ", sorted(df_offres_periodes["codePeriode"].unique()))

print("Nombre de lignes par nomenclature : ", df_offres_periodes["codeNomenclature"].value_counts())


Dimensions à filtrage :  (56, 13)
Périodes conservées :  ['2023T1', '2023T2', '2023T3', '2023T4', '2024T1', '2024T2', '2024T3', '2024T4', '2025T1', '2025T2', '2025T3', '2025T4', '2026T1', '2026T2']
Nombre de lignes par nomenclature :  codeNomenclature
PE                  14
PE-CUMUL12MOIS      14
TOFF                14
TOFF-CUMUL12MOIS    14
Name: count, dtype: int64


Nous remarquons que le filtrage conserve 14 trimestres, de 2023T1 à 2026T2, soit 56 lignes : 14 lignes pour chacune des quatre nomenclatures.

Nous allons définir le térritoire car notre périmètre est la France métropolitaine. 
Le code NAT désigne un niveau national, mais nous ne devons pas supposer qu'il exclut les territoires ultramarins sans vérifier la documentation.

France Travail distingue bien France métropolitaine et France entière dans son service Data Emploi. Nous devons donc identifier le code exact accepté par notre API avant de collecter les données.

Ainsi nous allons vérifier la structure des paramètres de notre API pour une autre région afin de vérifier l'équivalence et la cohérence avant la collecte.


In [118]:
# Nous allons vérifier le code géographique
# Nous allons vérifier les régions diponibles



In [119]:

import requests

url = (
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres"
)

# Nous allons tester pour la région 11 l'ile de france
payload = {
    "codeTypeTerritoire": "REG",
    "codeTerritoire": "11",
    "codeTypeActivite": "ROME",
    "codeActivite": "A1203",
    "codeTypePeriode": "TRIMESTRE",
    "codeTypeNomenclature": "ORIGINEOFF",
    "codeNomenclature": "TOFF"
}

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response_11 = requests.post(
    url,
    json=payload,
    headers=headers
)

print("Statut :", response_11.status_code)
print("Type de contenu :", response_11.headers.get("Content-Type"))
print("Réponse :", response_11.text[:3000])


Statut : 200
Type de contenu : application/json
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)","listeValeursParPeriode":[{"datMaj":"2026-09-23T09:06:22.000+02:00","codeTypeTerritoire":"REG","codeTerritoire":"11","libTerritoire":"ILE-DE-FRANCE","codeTypeActivite":"ROME","codeActivite":"A1203","libActivite":"Agent / Agente d'entretien des espaces verts","codeNomenclature":"PE","libNomenclature":"Offres France Travail","codeTypePeriode":"TRIMESTRE","codePeriode":"2026T2","libPeriode":"2ème trimestre 2026","valeurPrincipaleNombre":180,"valeurSecondairePourcentage":0.0,"listeValeurParCaract":[{"codeTypeCaract":"TYPECTR","codeCaract":"CDI","libCaract":"CDI","nombre":80,"pourcentage":47.0,"masque":false},{"codeTypeCaract":"TYPECTR","codeCaract":"CDD1","libCaract":"CDD < 1 mois","nombre

In [120]:
resultat_11 = response_11.json()

# Extraire les statistiques par périodes
valeurs_11 = resultat_11.get("listeValeursParPeriode", [])

df_offres_11 = pd.json_normalize(valeurs_11)

display(df_offres_11.head())

# Nous allons vérifier le libéllé du territoire

print("libTerritoire : ", df_offres_11["libTerritoire"][0])


,datMaj,codeTypeTerritoire,codeTerritoire,libTerritoire,codeTypeActivite,codeActivite,libActivite,codeNomenclature,libNomenclature,codeTypePeriode,codePeriode,libPeriode,valeurPrincipaleNombre,valeurSecondairePourcentage,listeValeurParCaract
0,2026-09-23T09:06:22.000+02:00,REG,11,ILE-DE-FRANCE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE,Offres France Travail,TRIMESTRE,2026T2,2ème trimestre 2026,180,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
1,2026-09-23T09:06:23.000+02:00,REG,11,ILE-DE-FRANCE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE-CUMUL12MOIS,Cumul 12 derniers mois offres France Travail,TRIMESTRE,2026T2,2ème trimestre 2026,660,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
2,2026-09-23T09:06:23.000+02:00,REG,11,ILE-DE-FRANCE,ROME,A1203,Agent / Agente d'entretien des espaces verts,TOFF,Toutes offres,TRIMESTRE,2026T2,2ème trimestre 2026,260,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
3,2026-09-23T09:06:23.000+02:00,REG,11,ILE-DE-FRANCE,ROME,A1203,Agent / Agente d'entretien des espaces verts,TOFF-CUMUL12MOIS,Cumul 12 derniers mois toutes offres,TRIMESTRE,2026T2,2ème trimestre 2026,920,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."
4,2026-09-23T09:06:22.000+02:00,REG,11,ILE-DE-FRANCE,ROME,A1203,Agent / Agente d'entretien des espaces verts,PE,Offres France Travail,TRIMESTRE,2026T1,1er trimestre 2026,220,0.0,"[{'codeTypeCaract': 'TYPECTR', 'codeCaract': '..."


libTerritoire :  ILE-DE-FRANCE


In [121]:
# Affichons la nomenclature de df_offres
print(df_offres_11.columns)
print(df_offres_11['codeNomenclature'].value_counts())

Index(['datMaj', 'codeTypeTerritoire', 'codeTerritoire', 'libTerritoire',
       'codeTypeActivite', 'codeActivite', 'libActivite', 'codeNomenclature',
       'libNomenclature', 'codeTypePeriode', 'codePeriode', 'libPeriode',
       'valeurPrincipaleNombre', 'valeurSecondairePourcentage',
       'listeValeurParCaract'],
      dtype='str')
codeNomenclature
PE                  22
PE-CUMUL12MOIS      22
TOFF                22
TOFF-CUMUL12MOIS    22
Name: count, dtype: int64


In [122]:
# Varifieer le territoire

df_offres_11[["codeTerritoire","libTerritoire"]].drop_duplicates()

,codeTerritoire,libTerritoire
0,11,ILE-DE-FRANCE


In [123]:
# Le nombre de métier ROME enregistrer
print(df_metiers_rome.columns)
print("Nombres de code métier : ", len(df_metiers_rome))
display(df_metiers_rome.head())

Index(['code', 'libelle'], dtype='str')
Nombres de code métier :  1923


,code,libelle
0,N2215,Coordonnateur / Coordonnatrice zone avion
1,N4207,Régulateur / Régulatrice poste central transpo...
2,L1321,Directeur / Directrice de production de specta...
3,I1639,Contrôleur / Contrôleuse de moteurs d'avions
4,N2217,Chef d'équipe contrôleur aérien / Cheffe d'équ...


In [124]:
# La liste des codes d'activité
df_metiers_rome["code"].tolist()

['N2215',
 'N4207',
 'L1321',
 'I1639',
 'N2217',
 'K1924',
 'K1923',
 'K1231',
 'K1429',
 'K1428',
 'K2515',
 'I1411',
 'C1124',
 'K1109',
 'K1111',
 'J1136',
 'J1139',
 'K2325',
 'K2416',
 'D1517',
 'M1228',
 'D1440',
 'D1444',
 'D1512',
 'I1511',
 'I1512',
 'C1125',
 'H2107',
 'N2103',
 'L1213',
 'L1212',
 'K2324',
 'F1712',
 'H2806',
 'D1254',
 'L1316',
 'L1318',
 'E1134',
 'L1314',
 'L1313',
 'H2106',
 'H3409',
 'H1410',
 'M1722',
 'L1319',
 'H2516',
 'K1611',
 'D1513',
 'J1141',
 'I1120',
 'H1311',
 'H1535',
 'L1320',
 'M1434',
 'M1721',
 'L1312',
 'K1741',
 'K2511',
 'B1816',
 'E1316',
 'H2511',
 'N2104',
 'L1214',
 'N4306',
 'N3111',
 'D1249',
 'D1439',
 'E1214',
 'L1315',
 'F1713',
 'A1220',
 'N4135',
 'I1210',
 'H2811',
 'H3411',
 'L1317',
 'L1310',
 'N4134',
 'K1739',
 'H2108',
 'K2326',
 'N4405',
 'K2414',
 'N4133',
 'H3410',
 'F1404',
 'L1216',
 'H2712',
 'H3308',
 'C1126',
 'G1246',
 'H3204',
 'H2711',
 'H2513',
 'D1250',
 'K1515',
 'H2514',
 'H1407',
 'H1313',
 'H2922',


In [125]:
# Nous allons tester de collecter pour 2 codes Activités pour éviter une grosse collecte 
payload_test = payload.copy()
payload_test["codeActivite"] = "D1512"
print(payload_test)

{'codeTypeTerritoire': 'REG', 'codeTerritoire': '11', 'codeTypeActivite': 'ROME', 'codeActivite': 'D1512', 'codeTypePeriode': 'TRIMESTRE', 'codeTypeNomenclature': 'ORIGINEOFF', 'codeNomenclature': 'TOFF'}


In [126]:
print(payload_test["codeTerritoire"])


11


In [127]:
url = (
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres"
)

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(
    url,
    json=payload_test,
    headers=headers
)

print("Statut API :", response.status_code)
print("Type de contenu :", response.headers.get("Content-Type"))
print("Réponse :", response.text[:1000])


Statut API : 200
Type de contenu : application/json
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)","listeValeursParPeriode":[{"datMaj":"2026-09-23T09:06:22.000+02:00","codeTypeTerritoire":"REG","codeTerritoire":"11","libTerritoire":"ILE-DE-FRANCE","codeTypeActivite":"ROME","codeActivite":"D1512","libActivite":"Directeur régional / Directrice régionale d'hypermarché ou de supermarché","codeNomenclature":"PE-CUMUL12MOIS","libNomenclature":"Cumul 12 derniers mois offres France Travail","codeTypePeriode":"TRIMESTRE","codePeriode":"2026T2","libPeriode":"2ème trimestre 2026","valeurPrincipaleNombre":0,"valeurSecondairePourcentage":0.0,"listeValeurParCaract":[{"codeTypeCaract":"TYPECTR","codeCaract":"CDI","libCaract":"CDI","nombre":0,"pourcentage":100.0,"masque":false},{"codeTypeCaract

In [128]:
# Convertissons la réponse actuelle en Dataframe
df_offres_test = pd.json_normalize(response.json()["listeValeursParPeriode"])

#display(df_offres_test)

# Observons les différentes péiodes er le territoire collecté

display(df_offres_test[["libPeriode", "codeTerritoire"]].drop_duplicates())
# Pour préparer la collecte des régions métropolitaines, nous allons vérifier les codes utilisés par l'API.


,libPeriode,codeTerritoire
0,2ème trimestre 2026,11
3,1er trimestre 2026,11
6,4ème trimestre 2025,11
9,3ème trimestre 2025,11
13,2ème trimestre 2025,11
17,1er trimestre 2025,11
20,4ème trimestre 2024,11
24,3ème trimestre 2024,11
27,2ème trimestre 2024,11
30,1er trimestre 2024,11


codeActivite accepte un seul code et pas une liste de code.

## Estimons le volume de notre collecte
Si nous collectons les 1 923 métiers dans les 12 régions métropolitaines, avec 2 nomenclatures (PE et TOFF), cela représenterait potentiellement :
- 1 923 métiers × 12 régions = 23 076 combinaisons métier-région.
- Si chaque requête renvoie les deux nomenclatures, environ 23 076 appels.
- Si une requête est nécessaire par nomenclature, environ 46 152 appels.

In [129]:
# Le nombre de lignes contenus dans ROME
nb_metiers = len(df_metiers_rome)
# La france métropolitaine compte 13 régions
nb_regions = 13

print("Nombre de métiers : ", nb_metiers)
print("Nombre de régions : ", nb_regions)
print("Nombre estimé d'appels : ", nb_metiers * nb_regions)


Nombre de métiers :  1923
Nombre de régions :  13
Nombre estimé d'appels :  24999


24 999 est le nombre d'appels estimé pour la collecte selon notre périmètre défini. Tout en sachant que certains métiers peuvent ne pas avoir de données pour certaines régions.


### Nous allons vérifier que les 13 régions sont bien accessibles avec leurs codes dans l'API France Travail.


In [130]:
# Les codes officielles des régions sont disponibles sur l'INSEE dans le code officiel géographique 2026

regions_metropolitaines = {
    "84": "AUVERGNE-RHONE-ALPES",
    "27": "BOURGOGNE-FRANCHE-COMTE",
    "53": "BRETAGNE",
    "24": "CENTRE-VAL DE LOIRE",
    "94": "CORSE",
    "44": "GRAND EST",
    "32": "HAUTS-DE-FRANCE",
    "11": "ILE-DE-FRANCE",
    "28": "NORMANDIE",
    "75": "NOUVELLE-AQUITAINE",
    "76": "OCCITANIE",
    "52": "PAYS DE LA LOIRE",
    "93": "PROVENCE-ALPES-COTE D'AZUR"
}

print("régions métropolitaines :", regions_metropolitaines)
print("nombre de régions : ", len(regions_metropolitaines))
      

régions métropolitaines : {'84': 'AUVERGNE-RHONE-ALPES', '27': 'BOURGOGNE-FRANCHE-COMTE', '53': 'BRETAGNE', '24': 'CENTRE-VAL DE LOIRE', '94': 'CORSE', '44': 'GRAND EST', '32': 'HAUTS-DE-FRANCE', '11': 'ILE-DE-FRANCE', '28': 'NORMANDIE', '75': 'NOUVELLE-AQUITAINE', '76': 'OCCITANIE', '52': 'PAYS DE LA LOIRE', '93': "PROVENCE-ALPES-COTE D'AZUR"}
nombre de régions :  13


In [131]:
# Véifions que l’API France Travail reconnaît ces codes,
# sachant que nous avaons déjà les régions 75 (Nouvelle Aquitaine) et 11 (Ile-de-France)
for code, region in regions_metropolitaines.items():
    payload_test["codeTerritoire"] = code
    response = requests.post(
        "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
        headers=headers,
        json=payload_test
    )

    if response.status_code == 200:
        data = response.json()["listeValeursParPeriode"]
        libelles = {
            ligne["libTerritoire"]
            for ligne in data}
        
        print(code,region,"->",libelles)
        
    else:
        print(code, region, "-> ERREUR", response.status_code)


84 AUVERGNE-RHONE-ALPES -> {'AUVERGNE-RHONE-ALPES'}
27 BOURGOGNE-FRANCHE-COMTE -> {'BOURGOGNE-FRANCHE-COMTE'}
53 BRETAGNE -> {'BRETAGNE'}
24 CENTRE-VAL DE LOIRE -> {'CENTRE-VAL DE LOIRE'}


KeyError: 'listeValeursParPeriode'

In [ ]:
print("Statut :", response.status_code)
print("Réponse :", response.text[:2000])

Statut : 200
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)"}


In [ ]:
print("Payload utilisé :", payload_test)
print("Réponse complète :", response.json())

Payload utilisé : {'codeTypeTerritoire': 'REG', 'codeTerritoire': '94', 'codeTypeActivite': 'ROME', 'codeActivite': 'D1512', 'codeTypePeriode': 'TRIMESTRE', 'codeTypeNomenclature': 'ORIGINEOFF', 'codeNomenclature': 'TOFF'}
Réponse complète : {'datMaj': '2026-09-23T11:04:21.000+02:00', 'codeIndicateur': 'OFF_1', 'codeFamille': 'OFFRES', 'libIndicateur': "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)"}


In [ ]:
payload_test["codeTerritoire"] = "94"
payload_test["codeActivite"] = "A1203"

response = requests.post(
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
    headers=headers,
    json=payload_test
)

print("Statut :", response.status_code)
print("Réponse :", response.json())

Statut : 200
Réponse : {'datMaj': '2026-09-23T11:04:21.000+02:00', 'codeIndicateur': 'OFF_1', 'codeFamille': 'OFFRES', 'libIndicateur': "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)", 'listeValeursParPeriode': [{'datMaj': '2026-09-23T09:06:21.000+02:00', 'codeTypeTerritoire': 'REG', 'codeTerritoire': '94', 'libTerritoire': 'CORSE', 'codeTypeActivite': 'ROME', 'codeActivite': 'A1203', 'libActivite': "Agent / Agente d'entretien des espaces verts", 'codeNomenclature': 'PE', 'libNomenclature': 'Offres France Travail', 'codeTypePeriode': 'TRIMESTRE', 'codePeriode': '2026T2', 'libPeriode': '2ème trimestre 2026', 'valeurPrincipaleNombre': 20, 'valeurSecondairePourcentage': 0.0, 'listeValeurParCaract': [{'codeTypeCaract': 'TYPECTR', 'codeCaract': 'CDI', 'libCaract': 'CDI', 'nombre': 10, 'pourcentage': 53.0, 'masque': False}, {'codeTypeCaract': 'TYPECTR', 'codeCaract': 'CDD1', 'libCaract': 'CDD < 1 mo

 On vérifie pourquoi ça bloque avec la CORSE, on va faire un test avec le métier qui fonctionnait déjà 'A1203'.
 Nous allons pour cela relancer la vérification des 13 régions avec A1203.
 

In [ ]:
for code, region in regions_metropolitaines.items():
    payload_test["codeTerritoire"] = code
    payload_test["codeActivite"] = "A1203"

    response = requests.post(
        "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
        headers=headers,
        json=payload_test
    )

    resultat = response.json()

    if response.status_code != 200:
        print(code, region, "-> ERREUR HTTP", response.status_code)
    elif "listeValeursParPeriode" not in resultat:
        print(code, region, "-> AUCUNE DONNÉE RETOURNÉE")
    else:
        libelles = {
            ligne["libTerritoire"]
            for ligne in resultat["listeValeursParPeriode"]
        }
        print(code, region, "->", libelles)

84 AUVERGNE-RHONE-ALPES -> {'AUVERGNE-RHONE-ALPES'}
27 BOURGOGNE-FRANCHE-COMTE -> {'BOURGOGNE-FRANCHE-COMTE'}
53 BRETAGNE -> {'BRETAGNE'}
24 CENTRE-VAL DE LOIRE -> {'CENTRE-VAL DE LOIRE'}
94 CORSE -> {'CORSE'}
44 GRAND EST -> {'GRAND EST'}
32 HAUTS-DE-FRANCE -> {'HAUTS-DE-FRANCE'}
11 ILE-DE-FRANCE -> {'ILE-DE-FRANCE'}
28 NORMANDIE -> {'NORMANDIE'}
75 NOUVELLE-AQUITAINE -> {'NOUVELLE-AQUITAINE'}
76 OCCITANIE -> {'OCCITANIE'}
52 PAYS DE LA LOIRE -> {'PAYS DE LA LOIRE'}
93 PROVENCE-ALPES-COTE D'AZUR -> {"PROVENCE-ALPES-COTE D'AZUR"}


Nous voyons que le test confirme que les 13 régions métropolitaines sont correctement reconnues par l’API France Travail avec le métier A1203.


### Testons la collecte d'un échantillon de métiers vue le nombre d'appel à venir

Ce test utilise les 10 premiers métiers sur les 13 régions du référentiel. Nous pourrons choisir un échantillon plus pertinent ensuite.
Ainsi nous aurons 130 requêtes ce qui nous permettra d'observer la disponibilité des données sans tout lancer.



In [ ]:
# Sélectionnons les 10 premiers métiers
echantillon_metiers = df_metiers_rome.head(10)
# la collecte sera stockée dans une liste
resultats_test = []

for code_metier in echantillon_metiers["code"] :
    for code_region, nom_region in regions_metropolitaines.items():
        payload_test["codeTerritoire"] = code_region
        payload_test["codeActivite"] = code_metier

        response = requests.post(
            "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
            headers=headers,
            json=payload_test
        )

        resultat = response.json()

        resultats_test.append({
            "code_metier" : code_metier,
            "code_region" : code_region,
            "region" : nom_region,
            "statut_http" : response.status_code,
            "donnees_presentes" : (
                "listeValeursParPeriode" in resultat
                and len(resultat.get("listeValeursParPeriode", [])) > 0
            )

        })
# Convertissons le résultat en dataframe

df_test_collecte = pd.DataFrame(resultats_test)

# affichons l'entête et quelques informations 

display(df_test_collecte.head())

print("Nombre de requêtes : ", len(df_test_collecte))
print("Réponses HTTP 200 : ", ((df_test_collecte["statut_http"] == 200).sum()))
print("Réponses avec données : ", ((df_test_collecte["donnees_presentes"]).sum()))
print("Réponses sans données : ", ((~df_test_collecte["donnees_presentes"]).sum()))


,code_metier,code_region,region,statut_http,donnees_presentes
0,N2215,84,AUVERGNE-RHONE-ALPES,404,False
1,N2215,27,BOURGOGNE-FRANCHE-COMTE,404,False
2,N2215,53,BRETAGNE,404,False
3,N2215,24,CENTRE-VAL DE LOIRE,404,False
4,N2215,94,CORSE,404,False


Nombre de requêtes :  130
Réponses HTTP 200 :  0
Réponses avec données :  0
Réponses sans données :  130


In [ ]:
# vérifions de manière individuel la requête du code_metier N2215 pour la région code 84
payload_test["codeTerritoire"] = "84"
payload_test["codeActivite"] = "N2215"

response = requests.post(
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
    headers=headers,
    json=payload_test
)

print("Statut HTTP :", response.status_code)
print("URL appelée :", response.url)
print("Type de contenu :", response.headers.get("Content-Type"))
print("Réponse :", response.text[:1000])

Statut HTTP : 404
URL appelée : https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres
Type de contenu : application/json
Réponse : {"message":"[FiltreErreurSldng] Erreur  lors de l'appel de l'Api REST 'POST:http://da084.z8-prod-mop.k8s.pole-emploi.intra/da084-smartemploi-sldng/v1/indicateur/stat-offres'. Status code = 404, message = Not Found","codeHttp":404}


In [ ]:
# Vérifions avec e code activité qui fonctionne déjà
payload_test["codeTerritoire"] = "84"
payload_test["codeActivite"] = "A1203"

response = requests.post(
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
    headers=headers,
    json=payload_test
)

print("Statut HTTP :", response.status_code)
print("Réponse :", response.text[:500])

Statut HTTP : 200
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)","listeValeursParPeriode":[{"datMaj":"2026-09-23T09:06:22.000+02:00","codeTypeTerritoire":"REG","codeTerritoire":"84","libTerritoire":"AUVERGNE-RHONE-ALPES","codeTypeActivite":"ROME","codeActivite":"A1203","libActivite":"Agent / Agente d'entretien des e


In [ ]:
df_metiers_rome.head()

,code,libelle
0,N2215,Coordonnateur / Coordonnatrice zone avion
1,N4207,Régulateur / Régulatrice poste central transpo...
2,L1321,Directeur / Directrice de production de specta...
3,I1639,Contrôleur / Contrôleuse de moteurs d'avions
4,N2217,Chef d'équipe contrôleur aérien / Cheffe d'équ...


In [ ]:
# Nous allons tester un petit groupe de métiers pour la région 84
codes_test = ["A1203", "N2215", "N4207", "I1639", "N2217"]

for code in codes_test:
    payload_test["codeTerritoire"] = "84"
    payload_test["codeActivite"] = code

    response = requests.post(
        "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres",
        headers=headers,
        json=payload_test
    )

    print(
        code,
        "-> HTTP", response.status_code,
        "|", response.text[:180]
    )

A1203 -> HTTP 200 | {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractér
N2215 -> HTTP 404 | {"message":"[FiltreErreurSldng] Erreur  lors de l'appel de l'Api REST 'POST:http://da084.z8-prod-mop.k8s.pole-emploi.intra/da084-smartemploi-sldng/v1/indicateur/stat-offres'. Statu
N4207 -> HTTP 404 | {"message":"[FiltreErreurSldng] Erreur  lors de l'appel de l'Api REST 'POST:http://da084.z8-prod-mop.k8s.pole-emploi.intra/da084-smartemploi-sldng/v1/indicateur/stat-offres'. Statu
I1639 -> HTTP 404 | {"message":"[FiltreErreurSldng] Erreur  lors de l'appel de l'Api REST 'POST:http://da084.z8-prod-mop.k8s.pole-emploi.intra/da084-smartemploi-sldng/v1/indicateur/stat-offres'. Statu
N2217 -> HTTP 404 | {"message":"[FiltreErreurSldng] Erreur  lors de l'appel de l'Api REST 'POST:http://da084.z8-prod-mop.k8s.pole-emploi.intra/da084-smartemploi-sldng/v1/indicateur/stat-offres'. S

In [134]:
codes_test = ["A1203", "N2215", "N4207", "I1639", "N2217"]

for code in codes_test:
    payload = {
        "codeTypeTerritoire": "REG",
        "codeTerritoire": "84",
        "codeTypeActivite": "ROME",
        "codeActivite": code,
        "codeTypePeriode": "TRIMESTRE",
        "codeTypeNomenclature": "ORIGINEOFF",
        "codeNomenclature": "TOFF"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    print(code, "-> HTTP", response.status_code)

A1203 -> HTTP 200
N2215 -> HTTP 404
N4207 -> HTTP 404
I1639 -> HTTP 404
N2217 -> HTTP 404
